# Finetuning LLM - Bhagwad Gita

In [10]:
!pip install unsloth

In [11]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from datasets import load_dataset
from datasets import Dataset
from transformers import TrainingArguments
import os
import pandas as pd
import re

### Loading a base model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


### Adding LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

### Loading the dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rambo011/bhagavad-gita-q-and-a-dataset-for-modern-life-problem")

print("Path to dataset files:", path)

In [ ]:
for f in os.listdir(path):

    print(f)

In [ ]:
files = [
    "Chapter_18_QA.csv",
"Chapter_5_QA.csv",
"Chapter_13_QA.csv",
"Chapter_10_QA.csv",
"Chapter_14_QA.csv",
"Chapter_15_QA.csv",
"Chapter_16_QA.csv",
"Chapter_11_QA.csv",
"Chapter_3_QA.csv",
"Chapter_7_QA.csv",
"Chapter_8_QA.csv",
"Chapter_1_QA.csv",
"Chapter_4_QA.csv",
"Chapter_9_QA.csv",
"Chapter_2_QA.csv",
"Chapter_17_QA.csv",
"Chapter_12_QA.csv",
"Chapter_6_QA.csv",

]
data = pd.concat(
    [pd.read_csv(os.path.join(path, f)) for f in files],
    ignore_index=True
)


len(data)

In [ ]:
data = data[['question','answer']]
data

In [ ]:
dataset = data

### Train

In [ ]:
def format_chat(sample):
    return {
        "messages": [
            {"role": "user", "content": sample["question"]},
            {"role": "assistant", "content": sample["answer"]}
        ]
    }

# Convert the pandas DataFrame 'dataset' into a 'datasets.Dataset' object
dataset = Dataset.from_pandas(dataset)

# Now, apply the formatting function. The .map() method of datasets.Dataset works as expected
dataset = dataset.map(format_chat)

def apply_template(sample):
    return {
        "text": tokenizer.apply_chat_template(
            sample["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

dataset = dataset.map(apply_template)

In [ ]:
dataset

In [ ]:
print(dataset[0]["text"])

In [ ]:
print(apply_template(dataset[5])["text"])

In [ ]:
trainer = SFTTrainer(model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    max_seq_length = 2048,
    # formatting_func = apply_template, # Use the new formatting function
    args = TrainingArguments(
        per_device_train_batch_size = 2, # Number of examples processed at once by the GPU
        gradient_accumulation_steps = 4,  # Accumulate gradients for 4 batches before updating the weights
                                          # Effective batch size = 2 × 4 = 8 examples
        warmup_ratio = 0.05,
        max_steps = -1,        # Total number of optimizer/update steps NOT number of epochs
        num_train_epochs = 1,
        learning_rate = 2e-4,  # learning rate
        output_dir = "outputs",
        save_steps=500,
        assistant_only_loss=True,

        logging_steps = 10,     # printing the output after N steps
    ),
)

trainer.train()

### Testing Fine Tuned Model

In [ ]:
FastLanguageModel.for_inference(model)

In [ ]:
messages = [
    {
        "role": "user",
        "content": "why am i always confused?"
    },
]


inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(inputs, max_new_tokens=100,
                         temperature=0.3,
                         top_p=0.9,
                         do_sample=True)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Saving the Model

In [ ]:
# Not merged with base model
# model.save_pretrained("Gita_finetuned_model")
# tokenizer.save_pretrained("Gita_finetuned_model")

# Merged with base model
model.save_pretrained_merged(
    "Gita_Qwen_2B",
    tokenizer,
    save_method="merged_16bit",
)

In [ ]:
print(os.path.exists("./Gita_Qwen_2B"))
print(os.listdir("./Gita_Qwen_2B"))

In [ ]:
merged_model, merged_tokenizer = FastLanguageModel.from_pretrained(
    model_name="./Gita_Qwen_2B",
    max_seq_length=2048,
    load_in_4bit=True,
)


In [ ]:
inputs = merged_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = merged_model.generate(inputs,
                         max_new_tokens=100,
                         temperature=0.01,
                         top_p=0.9,
                         do_sample=True)

print(
    merged_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

### Comparing to Instruct model

In [ ]:
instruct_model, instruct_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

In [ ]:
messages = [
    {
        "role": "user",
        "content": "Why am i always confused?"
    }
]

inputs = instruct_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = instruct_model.generate(
    inputs,
    max_new_tokens=100
)

print(instruct_tokenizer.decode(outputs[0], skip_special_tokens=True))

### Saving on HF

In [ ]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_UPLOAD")
login(token=token)

In [ ]:
from huggingface_hub import HfApi


api = HfApi()

api.create_repo(
    repo_id="Nikhila15/Gita-Qwen-2B",
    repo_type="model",
    exist_ok=True,
)

api.upload_folder(
    folder_path="Gita_Qwen_2B",
    repo_id="Nikhila15/Gita-Qwen-2B",
    repo_type="model",
)